In [8]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 1. Загрузка и базовый сплит
df = pd.read_csv('../data/clean_data.csv')
print(f"Размер датасета: {df.shape}")
print("\nРаспределение целевой переменной (Is Rejected):")
print(df['Is Rejected'].value_counts())
print("\nДоли классов:")
print((df['Is Rejected'].value_counts(normalize=True) * 100).round(4))


Размер датасета: (22135, 13)

Распределение целевой переменной (Is Rejected):
Is Rejected
0    12135
1    10000
Name: count, dtype: int64

Доли классов:
Is Rejected
0    54.8227
1    45.1773
Name: proportion, dtype: float64


In [9]:
X, y = df.drop(columns=['Is Rejected']), df['Is Rejected']

assert X.shape[0] == y.shape[0], "Размеры фичей и таргета не совпадают"

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

print("\n--- После Split ---")
print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print("\nДоли таргета в Train:")
print((y_train.value_counts(normalize=True) * 100).round(4).to_dict())
print("Доли таргета в Test:")
print((y_test.value_counts(normalize=True) * 100).round(4).to_dict())




--- После Split ---
Train size: 17708 | Test size: 4427

Доли таргета в Train:
{0: 54.8227, 1: 45.1773}
Доли таргета в Test:
{0: 54.8227, 1: 45.1773}


In [10]:
# 2. Настройка препроцессоров
num_cols = X_train.select_dtypes(include=['number']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

LR_DROP_COLS = ['Age', 'Credit History']
skewed_cols = [c for c in ['Employee Experience', 'Person Income', 'Loan Amount', 'Loan percentage'] if c in num_cols and c not in LR_DROP_COLS]
normal_cols = [c for c in num_cols if c not in skewed_cols + LR_DROP_COLS]
num_cols_raw = [c for c in num_cols if c not in LR_DROP_COLS]



In [11]:
cat_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

num_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

log_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, validate=False)),
    ('scaler', StandardScaler())
])

tree_num = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipe_native = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent'))
])



In [12]:
prep_tree = ColumnTransformer(transformers=[
    ('num', tree_num, num_cols),
    ('cat', cat_pipe, cat_cols)
])

prep_raw = ColumnTransformer(transformers=[
    ('num', num_pipe, num_cols_raw),
    ('cat', cat_pipe, cat_cols)
])

prep_log = ColumnTransformer(transformers=[
    ('skewed', log_pipe, skewed_cols),
    ('normal', num_pipe, normal_cols),
    ('cat', cat_pipe, cat_cols)
])

prep_cb_native = ColumnTransformer(transformers=[
    ('num', tree_num, num_cols),
    ('cat', cat_pipe_native, cat_cols)
])

cb_cat_indices = list(range(len(num_cols), len(num_cols) + len(cat_cols)))



In [13]:
# 3. Сборка бейзлайнов
models = {
    "Dummy Classifier": Pipeline(steps=[
        ('preprocessor', prep_tree),
        ('model', DummyClassifier(strategy='prior'))
    ]),
    "LogReg (Raw)": Pipeline(steps=[
        ('preprocessor', prep_raw),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42, n_jobs=-1))
    ]),
    "LogReg (Log)": Pipeline(steps=[
        ('preprocessor', prep_log),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42, n_jobs=-1))
    ]),
    "LightGBM": Pipeline(steps=[
        ('preprocessor', prep_tree),
        ('model', LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1, n_jobs=-1))
    ]),
    # ЭКСПЕРИМЕНТ A: CatBoost с One-Hot Encoding
    "CatBoost (OHE)": Pipeline(steps=[
        ('preprocessor', prep_tree),
        ('model', CatBoostClassifier(auto_class_weights='Balanced', random_state=42, verbose=False, thread_count=-1))
    ]),
    
    # ЭКСПЕРИМЕНТ B: CatBoost с нативными категориями
    "CatBoost (Native)": {
        "pipeline": Pipeline(steps=[
            ('preprocessor', prep_cb_native),
            ('model', CatBoostClassifier(
                auto_class_weights='Balanced', 
                random_state=42, 
                verbose=False, 
                thread_count=2
            ))
        ]),
        "fit_params": {'model__cat_features': cb_cat_indices}
    }
}


In [14]:
# 4. Функция оценки с автоматическим парсингом метрик
def evaluate_models(models_dict, X, y):
    scoring = {'roc_auc': 'roc_auc', 'pr_auc': 'average_precision', 'accuracy': 'accuracy', 
               'precision': 'precision', 'recall': 'recall', 'f1': 'f1'}
    results = []
    
    for name, config in models_dict.items():
        if isinstance(config, dict):
            pipeline = config['pipeline']
            fit_params = config.get('fit_params', None)
        else:
            pipeline = config
            fit_params = None
        scores = cross_validate(
            pipeline, X, y, cv=5, 
            scoring=scoring, 
            params=fit_params, 
            n_jobs=-1
        )
        model_res = {'Model': name,
            'ROC_AUC': scores['test_roc_auc'].mean(),
            'ROC_AUC_STD': scores['test_roc_auc'].std()}
        
        model_res.update({k.replace('test_', '').upper(): v.mean() for k, v in scores.items() if 'test_' in k})
        results.append(model_res)
        
    return pd.DataFrame(results).sort_values(by='ROC_AUC', ascending=False)

cv_results = evaluate_models(models, X_train, y_train)
print(cv_results.to_string(index=False))

            Model  ROC_AUC  ROC_AUC_STD   PR_AUC  ACCURACY  PRECISION   RECALL       F1
   CatBoost (OHE) 0.933945     0.002836 0.936670  0.861645   0.875152 0.809500 0.840914
         LightGBM 0.933313     0.002857 0.935483  0.859160   0.874448 0.803750 0.837555
CatBoost (Native) 0.933197     0.002355 0.935896  0.862209   0.878838 0.806375 0.840938
     LogReg (Log) 0.874355     0.001431 0.862022  0.791620   0.758424 0.790875 0.774257
     LogReg (Raw) 0.868122     0.001015 0.855459  0.785408   0.753038 0.781375 0.766902
 Dummy Classifier 0.500000     0.000000 0.451773  0.548227   0.000000 0.000000 0.000000


LightGBM и обе версии CatBoost демонстрируют практически идентичное качество. Разница в среднем ROC-AUC между лучшей и худшей моделью составляет менее 0.001, что не позволяет считать одну из моделей однозначно превосходящей другие на данном этапе.

Выбор конкретного алгоритма градиентного бустинга в данном случае оказывает значительно меньшее влияние на качество, чем сам переход от линейной модели к нелинейным ансамблевым моделям.

In [15]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

# 1. Берем пайплайн LightGBM с фиксированным именем шага 'model'
lgbm_pipeline = Pipeline(steps=[
    ('preprocessor', prep_tree),
    ('model', LGBMClassifier(
        objective='binary',
        class_weight='balanced',
        subsample_freq=1,     
        random_state=42,
        verbose=-1,
        n_jobs=-1
    ))
])

# 2. Сетка параметров с префиксом 'model__'
param_distributions = {
    'model__n_estimators': randint(100, 1000),
    'model__learning_rate': loguniform(0.005, 0.2),
    'model__num_leaves': randint(15, 127),
    'model__max_depth': [-1, 3, 5, 7, 9, 12, 15],
    'model__min_child_samples': randint(10, 100),
    'model__subsample': uniform(0.5, 0.5),       
    'model__colsample_bytree': uniform(0.5, 0.5), 
    'model__reg_alpha': loguniform(1e-3, 10.0),
    'model__reg_lambda': loguniform(1e-3, 10.0)
}

# 3. Настройка RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=param_distributions,
    n_iter=50,             
    scoring='roc_auc',     
    cv=5,                 
    random_state=42,
    n_jobs=-1,
    verbose=1
)

# 4. Запуск поиска
random_search.fit(X_train, y_train)

# 5. Вывод результатов
print(f"\nЛучший ROC-AUC на CV: {random_search.best_score_:.4f}")
print("\nОптимальные гиперпараметры:")
for param, value in random_search.best_params_.items():
    print(f"  {param.replace('model__', '')}: {value}")

# Сохранение лучшей модели
best_lgbm_model = random_search.best_estimator_

Fitting 5 folds for each of 50 candidates, totalling 250 fits

Лучший ROC-AUC на CV: 0.9386

Оптимальные гиперпараметры:
  colsample_bytree: 0.7248770666848828
  learning_rate: 0.0214795108988544
  max_depth: 7
  min_child_samples: 23
  n_estimators: 962
  num_leaves: 62
  reg_alpha: 0.02023778954048508
  reg_lambda: 0.19132683954526483
  subsample: 0.7604171300129119


In [16]:
from sklearn.model_selection import RandomizedSearchCV

# 1. Достаем пайплайн для нативного CatBoost из словаря
cb_native_pipeline = models["CatBoost (Native)"]["pipeline"]

# 2. Задаем сетку гиперпараметров
cb_param_dist = {
    'model__iterations': [300, 500, 800],
    'model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'model__depth': [4, 6],
    'model__l2_leaf_reg': [1, 3, 5, 10],
    'model__random_strength': [0.1, 1, 5],
    'model__bootstrap_type': ['Bayesian'], # Обязательно для bagging_temperature
    'model__bagging_temperature': [0.0, 0.5, 1.0, 2.0]
}

# 3. Настраиваем рандомизированный поиск
cb_search = RandomizedSearchCV(
    estimator=cb_native_pipeline,
    param_distributions=cb_param_dist,
    n_iter=50,          
    scoring='roc_auc', 
    cv=5,
    random_state=42,
    n_jobs=-1,        
    verbose=2
)

# 4. Запускаем обучение
print("Начинаем подбор параметров для CatBoost Native...")
cb_search.fit(X_train, y_train, model__cat_features=cb_cat_indices)

print("\n--- Результаты тюнинга ---")
print(f"Лучший ROC_AUC на кросс-валидации: {cb_search.best_score_:.4f}")
print("Лучшие параметры:")
for param, value in cb_search.best_params_.items():
    print(f"  {param.replace('model__', '')}: {value}")

Начинаем подбор параметров для CatBoost Native...
Fitting 5 folds for each of 50 candidates, totalling 250 fits

--- Результаты тюнинга ---
Лучший ROC_AUC на кросс-валидации: 0.9339
Лучшие параметры:
  random_strength: 5
  learning_rate: 0.1
  l2_leaf_reg: 1
  iterations: 800
  depth: 6
  bootstrap_type: Bayesian
  bagging_temperature: 0.5


Я выбрал lightGBM для продолжения работы с проектом

In [17]:
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score

# 1. Создаем финальную модель LightGBM с лучшими гиперпараметрами
best_lgbm = random_search.best_estimator_



In [18]:
# 2. Обучаем модель на всех тренировочных данных
print("Обучение финальной модели...")
best_lgbm.fit(X_train, y_train)



Обучение финальной модели...


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [19]:
# проверяем все метрики на идеальной модели
scoring = {
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1'
}

print("Проведение кросс-валидации для идеальной модели...")
cv_scores = cross_validate(best_lgbm, X_train, y_train, cv=5, scoring=scoring, n_jobs=-1)

cv_train_results = pd.DataFrame({
    'Metric': ['ROC-AUC', 'PR-AUC', 'Accuracy', 'Precision', 'Recall', 'F1'],
    'CV Mean (Train)': [
        round(cv_scores['test_roc_auc'].mean(), 4),
        round(cv_scores['test_pr_auc'].mean(), 4),
        round(cv_scores['test_accuracy'].mean(), 4),
        round(cv_scores['test_precision'].mean(), 4),
        round(cv_scores['test_recall'].mean(), 4),
        round(cv_scores['test_f1'].mean(), 4)
    ],
    'CV Std': [
        round(cv_scores['test_roc_auc'].std(), 4),
        round(cv_scores['test_pr_auc'].std(), 4),
        round(cv_scores['test_accuracy'].std(), 4),
        round(cv_scores['test_precision'].std(), 4),
        round(cv_scores['test_recall'].std(), 4),
        round(cv_scores['test_f1'].std(), 4)
    ]
})

print("\n Результаты тестирования на X_train (5-fold CV):")
print(cv_train_results.to_string(index=False))

Проведение кросс-валидации для идеальной модели...

 Результаты тестирования на X_train (5-fold CV):
   Metric  CV Mean (Train)  CV Std
  ROC-AUC           0.9386  0.0030
   PR-AUC           0.9404  0.0028
 Accuracy           0.8667  0.0044
Precision           0.8822  0.0089
   Recall           0.8138  0.0093
       F1           0.8465  0.0051


In [20]:
# 3. Делаем предсказания на отложенной тестовой выборке
y_test_proba = best_lgbm.predict_proba(X_test)[:, 1]
y_test_pred = best_lgbm.predict(X_test)



In [21]:
# 4. Считаем все метрики
test_roc_auc = roc_auc_score(y_test, y_test_proba)
test_pr_auc = average_precision_score(y_test, y_test_proba)
test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred)
test_rec = recall_score(y_test, y_test_pred)
test_f1 = f1_score(y_test, y_test_pred)



In [22]:
# 5. Формируем финальную таблицу
final_results = pd.DataFrame({
    'Metric': ['ROC-AUC', 'PR-AUC', 'Accuracy', 'Precision', 'Recall', 'F1'],
    'CV (Train)': cv_train_results['CV Mean (Train)'].tolist(), 
    'Test (Unseen)': [
        round(test_roc_auc, 4),
        round(test_pr_auc, 4),
        round(test_acc, 4),
        round(test_prec, 4),
        round(test_rec, 4),
        round(test_f1, 4)
    ]
})

print("\nФинальные результаты:")
print(final_results.to_string(index=False))


Финальные результаты:
   Metric  CV (Train)  Test (Unseen)
  ROC-AUC      0.9386         0.9434
   PR-AUC      0.9404         0.9435
 Accuracy      0.8667         0.8735
Precision      0.8822         0.8883
   Recall      0.8138         0.8235
       F1      0.8465         0.8547


In [23]:
import joblib

# Сохраняем обученный финальный пайплайн в файл
joblib.dump(best_lgbm, 'best_lightgbm_pipeline.joblib')

print("Пайплайн успешно сохранен в файл 'best_lightgbm_pipeline.joblib'")

Пайплайн успешно сохранен в файл 'best_lightgbm_pipeline.joblib'
